In [1]:
import json, base64
from PIL import Image
from io import BytesIO
import os

notebook_name = "5.ipynb"
target_filename = "4fa6155f-5d68-4437-8f36-8e4ccc92dad8.png"
output_dir = "./images"

os.makedirs(output_dir, exist_ok=True)

with open(notebook_name, "r", encoding="utf-8") as f:
    notebook = json.load(f)

found = False
for cell in notebook.get("cells", []):
    attachments = cell.get("attachments", {})
    if target_filename in attachments:
        for mime, data in attachments[target_filename].items():
            try:
                img_data = base64.b64decode(data)
                img = Image.open(BytesIO(img_data))

                # Handle alpha channel safely
                if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
                    background = Image.new("RGB", img.size, (255, 255, 255))
                    background.paste(img, mask=img.split()[-1])
                    img = background
                else:
                    img = img.convert("RGB")

                # Show image before saving
                # img.show()

                # Save to next number
                existing = [int(f.split(".")[0]) for f in os.listdir(output_dir) if f.endswith(".jpg") and f.split(".")[0].isdigit()]
                name = max(existing) + 1 if existing else 1
                img.save(f"{output_dir}/{name}.jpg", "JPEG", quality=95)
                print(f"✅ Saved as {name}.jpg")
                found = True
                break
            except Exception as e:
                print("❌ Error during image decode/save:", e)
                break
    if found:
        break

if not found:
    print("❌ Attachment not found.")


✅ Saved as 52.jpg


In [2]:
%%bash
git status

On branch main
nothing to commit, working tree clean


In [26]:
%%bash
cd ~/.ssh
ls

config
id_ed25519
id_ed25519.pub
id_rsa
id_rsa.pub
known_hosts


In [28]:
%%bash
cd ~/.ssh
cat id_rsa.pub

ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAACAQC6VQA9/+JcSdV2YJ/Dy+LUiIxXXvSMlOrSGd6p1I6+6Y4GZ4kvPMHTgjTYkEKx69dXWjGdAjo6BYZHgdo6/MwO+EFB1fRmCg/Nf6wjrFMPjSJALVjoIwFgRYFBhFCJzGRcazr17dsqQmsn7uRv6oKEYe53HTrVxrISm2HU6ALu42wUcc84etNk0PYRn3u4SHuSy0z64ZFBWdg3lb8V91YVEjsUe56PBivs8toR7wjwGzI2YiB3IJ10BvpkwG+2xEsi8OYSUYs7fG7d7Hc/7pK9f5LakD5v9wRHcybVmiclRa2BDSpUMIJv87dcut1+T9YZjDQGPdc3vV1kg/x1nPdjfLlLWeFcWDj+GBlkWjNM3maueRlCk2ADq+csx+fGBK7UsufT/jsoUgmwa25YfXq+aNozXq9MDuI8yR+FXhWfh+MDrXlVEby0RgonqhidgXtzI2fcX1z0zU7PV6OQd6YAI8zRide97w0kNofuahqlmvWHbJ+6woFvdUHOpajdWafQAYC4K2JzdprRA7ztRtWEqSNQBM9D7PAdxFXviP6bWxwAjILaXt4VbWYAI/FgVr49oTFs+qG43TOlKLaPY5ADfq8RtXs3DrGX2W3lt6GBzi4Ar8e6LYqWiYXgFHCiY+LgZcahY6gEDvFqd1V+GjLZw3cUeFUdch2VKuG4HOtyYw== arch@github


In [36]:
%%bash
cd ~/.ssh
cat <<< """Host github-arch
    HostName github.com
    User git
    IdentityFile ~/.ssh/id_rsa_arch
    IdentitiesOnly yes""" > config

In [37]:
%%bash
cd ~/.ssh
cat config

Host github-arch
    HostName github.com
    User git
    IdentityFile ~/.ssh/id_rsa_arch
    IdentitiesOnly yes


In [43]:
%%bash
ssh -T git@github-arch

no such identity: /home/mubeen/.ssh/id_rsa_arch: No such file or directory
git@github.com: Permission denied (publickey).


CalledProcessError: Command 'b'ssh -T git@github-arch\n'' returned non-zero exit status 255.